# Kémzy PersonaLive CUDA Proof v6

Run **all cells from top to bottom**. This notebook is the GitHub source of truth for the Kaggle proof: it clones the pinned PersonaLive commit, applies the committed compatibility patch, reuses already-mounted weights when available, prepares the known source/driving inputs, disables xFormers, and runs the real offline renderer. It never downloads the 10+ GB PersonaLive checkpoints automatically.

In [ ]:
import os, sys, subprocess, json, pathlib, shutil, time
print('Python:', sys.executable)
print('Version:', sys.version.split()[0])
assert os.path.exists('/kaggle/working'), 'This notebook must run on Kaggle.'
print('Kaggle working directory: PASS')

In [ ]:
import subprocess, os, pathlib
ROOT='/kaggle/working/PersonaLive'
if os.path.isdir(ROOT):
    print('PersonaLive directory already exists; refreshing only if it is not the pinned commit.')
    r=subprocess.run(['git','-C',ROOT,'rev-parse','HEAD'],capture_output=True,text=True)
    if r.returncode==0 and r.stdout.strip()=='abdd112e01dcf7d89122c2e5efa29fcff0669740':
        print('Pinned PersonaLive checkout already present.')
    else:
        shutil.rmtree(ROOT)
if not os.path.isdir(ROOT):
    subprocess.run(['git','clone','https://github.com/GVCLab/PersonaLive.git',ROOT],check=True)
    subprocess.run(['git','-C',ROOT,'checkout','--detach','abdd112e01dcf7d89122c2e5efa29fcff0669740'],check=True)
head=subprocess.check_output(['git','-C',ROOT,'rev-parse','HEAD'],text=True).strip()
assert head=='abdd112e01dcf7d89122c2e5efa29fcff0669740', head
print('PersonaLive pinned commit:', head)

In [ ]:
import subprocess, pathlib, os
KEMZY='/kaggle/working/Kemzy-LiveAvatar'
if not os.path.isdir(KEMZY):
    subprocess.run(['git','clone','--branch','feature/backend-render-gateway','https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git',KEMZY],check=True)
else:
    subprocess.run(['git','-C',KEMZY,'fetch','origin','feature/backend-render-gateway'],check=True)
    subprocess.run(['git','-C',KEMZY,'reset','--hard','origin/feature/backend-render-gateway'],check=True)
print(subprocess.check_output(['git','-C',KEMZY,'rev-parse','HEAD'],text=True).strip())
patch=pathlib.Path(KEMZY)/'tools/personalive_compat_patch.py'
assert patch.is_file(), patch
subprocess.run([sys.executable,str(patch),ROOT],check=True)
print('Committed compatibility patch applied.')

In [ ]:
import importlib, subprocess, sys
required={'mediapipe':'0.10.13','av':'18.1.0','decord':'0.6.0'}
missing=[]
for mod,want in required.items():
    try:
        m=importlib.import_module(mod)
        got=getattr(m,'__version__',None)
        print(mod, got)
        if got != want: missing.append(f'{mod}=={want}')
    except Exception:
        missing.append(f'{mod}=={want}')
if missing:
    print('Installing only:', missing)
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
else:
    print('Pinned runtime dependencies already present.')
import torch, diffusers
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
print('Diffusers:',diffusers.__version__)
assert torch.cuda.is_available(), 'CUDA GPU is required.'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os, shutil
PR=Path(ROOT)/'pretrained_weights'
PR.mkdir(parents=True,exist_ok=True)
required_files=[
 'personalive/denoising_unet.pth','personalive/motion_encoder.pth','personalive/motion_extractor.pth','personalive/pose_guider.pth','personalive/reference_unet.pth','personalive/temporal_module.pth',
 'sd-vae-ft-mse/diffusion_pytorch_model.bin','sd-vae-ft-mse/config.json',
 'sd-image-variations-diffusers/image_encoder/pytorch_model.bin','sd-image-variations-diffusers/image_encoder/config.json',
 'sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin','sd-image-variations-diffusers/unet/config.json','sd-image-variations-diffusers/model_index.json']
EXPECTED_SIZES={
 'personalive/denoising_unet.pth':4927015578,
 'personalive/motion_encoder.pth':246719031,
 'personalive/motion_extractor.pth':112545505,
 'personalive/pose_guider.pth':4351700,
 'personalive/reference_unet.pth':3438324340,
 'personalive/temporal_module.pth':1817903019,
 'sd-vae-ft-mse/diffusion_pytorch_model.bin':334707217,
 'sd-vae-ft-mse/config.json':547,
 'sd-image-variations-diffusers/image_encoder/pytorch_model.bin':1215993867,
 'sd-image-variations-diffusers/image_encoder/config.json':703,
 'sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin':3438350225,
 'sd-image-variations-diffusers/unet/config.json':471,
 'sd-image-variations-diffusers/model_index.json':545,
}
def find_file(rel):
    target=Path(rel).name
    candidates=[]
    for base in [Path('/kaggle/input'),Path('/kaggle/working')]:
        if not base.exists(): continue
        for p in base.rglob(target):
            if p.is_file() and p.stat().st_size==EXPECTED_SIZES.get(rel,p.stat().st_size):
                candidates.append(p)
    if len(candidates)>1:
        print('Multiple size-matching candidates for',rel)
    return candidates[0] if candidates else None
missing=[]
for rel in required_files:
    dst=PR/rel
    if dst.is_file(): continue
    src=find_file(rel)
    if src is None:
        missing.append(rel); continue
    dst.parent.mkdir(parents=True,exist_ok=True)
    dst.symlink_to(src)
    print('linked',rel,'<-',src)
if missing:
    raise RuntimeError('Required model files are not mounted in Kaggle. No automatic multi-GB download was attempted:\n'+'\n'.join(missing))
print('All required PersonaLive/base/VAE files are present.')

In [ ]:
from pathlib import Path
import shutil
demo=Path(ROOT)/'demo'; demo.mkdir(exist_ok=True)
def first_matching(names):
    for base in [Path('/kaggle/input'),Path('/kaggle/working')]:
        if base.exists():
            for n in names:
                for p in base.rglob(n):
                    if p.is_file() and '/PersonaLive/' not in str(p): return p
    return None
img=Path('/kaggle/input/datasets/seravellenyrovalen/kemzyphoto/7322a267d7716f19b6792bf610fb0961.jpg')
if not img.is_file(): img=first_matching(['*.jpg','*.jpeg','*.png'])
vid=first_matching(['driving_video.mp4'])
if vid is None:
    vid=first_matching(['*.mp4'])
assert img is not None and img.is_file(), 'Reference image not found in Kaggle input.'
assert vid is not None and vid.is_file(), 'Driving video not found in Kaggle input.'
shutil.copy2(img,demo/'ref_img.jpg')
shutil.copy2(vid,demo/'driving_video.mp4')
print('Reference:',img, img.stat().st_size)
print('Driving:',vid, vid.stat().st_size)

In [ ]:
import sys, pathlib, torch
sys.path.insert(0,ROOT)
from src.models.motion_encoder.encoder import MotEncoder
m=MotEncoder().eval()
pe=m.pe
print('MotionEncoder PE shape:',tuple(pe.shape))
assert tuple(pe.shape)==(1,32,16), tuple(pe.shape)
print('MotionEncoder compatibility: PASS')

In [ ]:
import subprocess, os, pathlib, time
out=pathlib.Path(ROOT)/'results'
if out.exists(): shutil.rmtree(out)
cmd=[sys.executable,'inference_offline.py','--config','configs/prompts/personalive_offline.yaml','--name','kemzy_cuda_proof_v6','-W','512','-H','512','-L','4','--device','cuda','--reference_image',str(demo/'ref_img.jpg'),'--driving_video',str(demo/'driving_video.mp4')]
print('Running real PersonaLive inference with xFormers disabled by the committed patch...')
started=time.time()
subprocess.run(cmd,cwd=ROOT,check=True)
print('Inference seconds:',round(time.time()-started,2))

In [ ]:
from pathlib import Path
mp4s=sorted((Path(ROOT)/'results').rglob('*.mp4'))
print('Generated videos:',[(str(p),p.stat().st_size) for p in mp4s])
assert mp4s and all(p.stat().st_size>1000 for p in mp4s), 'No non-empty generated PersonaLive video found.'
proof=Path('/kaggle/working/kemzy_personalive_cuda_proof_v6.txt')
proof.write_text('PersonaLive CUDA proof PASS\ncommit=abdd112e01dcf7d89122c2e5efa29fcff0669740\nrenderer_output='+str(mp4s[0])+'\nsize='+str(mp4s[0].stat().st_size)+'\n',encoding='utf-8')
print(proof.read_text())

In [ ]:
# Optional backend smoke-test: import the Kémzy gateway without starting a second long-lived server.
import sys
sys.path.insert(0,KEMZY)
import backend.renderer.personalive_server as gateway
gateway.APP_ARGS=gateway.build_args()
print('Gateway import: PASS')
print('Gateway acceleration:',gateway.APP_ARGS.acceleration)
assert gateway.APP_ARGS.acceleration=='none'
print('Backend configuration: PASS — ACCELERATION=none')